# Кросс-валидация на реальных данных

Учебный пример к **занятию 5**: различные способы построения тестовых выборок.

Данные — наблюдения метеостанции **Малые Кармакулы** (таблица подготовлена на втором занятии, `Lect02_EDA_demo.ipynb`). Задача: восстановить температуру воздуха `ta` по остальным метеорологическим величинам.

План работы:

1. посмотреть на «наивную» оценку — обучение и проверка на одних и тех же данных;
2. познакомиться с кросс-валидацией штатными средствами `scikit-learn` (`cross_val_score`, `cross_validate`);
3. разобраться, как устроено разбиение на тренировочную и тестовую части и зачем группировать наблюдения по сезонам;
4. написать собственный генератор разбиений (blocked k-fold) для временных рядов.

Полезные ссылки:
- [Пайплайны в sklearn (документация)](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html)
- [Пайплайны в sklearn (youtube)](https://www.youtube.com/watch?v=jzKSAeJpC6s)
- [Кросс-валидация в sklearn (на русском)](https://scikit-learn.ru/stable/modules/cross_validation.html)

Дополнительные материалы:
- [Кросс-валидация (Яндекс.образование)](https://education.yandex.ru/handbook/ml/article/kross-validaciya)
- [Data Splitting Strategies](https://amueller.github.io/aml/04-model-evaluation/1-data-splitting-strategies.html)
- [Resampling Approaches](https://allmodelsarewrong.github.io/resampling.html)


## Импорт библиотек

Подключаем числовые и табличные инструменты, графику, индикатор прогресса `tqdm`, а также пайплайн, масштабирование, разбиения выборки, метрики и линейную регрессию из `scikit-learn`.

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [tqdm](https://tqdm.github.io/), [sklearn.linear_model.LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [sklearn.model_selection](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.model_selection), [sklearn.preprocessing.StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [sklearn.pipeline.Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import  tqdm

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error


## Подготовка данных

И снова здравствуйте, Малые Кармакулы

Читаем таблицу, подготовленную на втором занятии (`Lect02_EDA_demo.ipynb`), и повторяем минимальную чистку: убираем `tg` (температура поверхности почвы — почти вся отсутствует), `td` и `psl` (дублируют другие признаки), затем удаляем строки с пропусками и сравниваем размер таблицы до и после.

**Инструменты:** [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [DataFrame.drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html), [DataFrame.dropna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html), [DataFrame.shape](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shape.html), [DataFrame.head](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html).


In [ ]:
df = pd.read_csv('EDA_demo_data/karmakuly_processed.csv', parse_dates=['Date'], index_col='Date')
df.head()

#df = df['2015':]

print ('df shape with nans: ', df.shape)
df = df.drop (['tg', 'td', 'psl'], axis=1)
df = df.dropna()
print ('df shape without nans: ', df.shape)
df.head()



## Порождение дополнительных признаков

Добавляем два «циклических» признака вместо «сырых» номера дня года и часа:

- `year_pos` — косинус фазы годового цикла: $\cos(\pi + 2\pi \cdot \text{dayofyear} / 365)$, то есть −1 в начале января и +1 в середине лета;
- `day_pos` — косинус фазы суточного цикла: $\cos(\pi + 2\pi \cdot \text{hour} / 24)$, то есть −1 в полночь и +1 в полдень.

У «сырых» `dayofyear` и `hour` есть разрыв на границе года и суток, а у косинуса — нет: модели проще выучить плавный сезонный и суточный ход. Строим два графика — годовой ход за 2015 год и суточный ход за первые три дня 2015 года.

**Инструменты:** [DatetimeIndex.dayofyear](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.dayofyear.html), [DatetimeIndex.hour](https://pandas.pydata.org/docs/reference/api/pandas.DatetimeIndex.hour.html), [numpy.cos](https://numpy.org/doc/stable/reference/generated/numpy.cos.html), [DataFrame.plot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html), [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).


In [ ]:
df['year_pos'] = np.cos (np.pi + 2 * np.pi * df.index.dayofyear/365)
df['day_pos']  = np.cos (np.pi + 2 * np.pi * df.index.hour/24)

fig, ax = plt.subplots (1,2, figsize=(10,5))

df['2015':]['year_pos'].plot(ax = ax[0], title='year_pos (annual cycle)',
                             xlabel='Дата', ylabel='year_pos (косинус фазы года)')
df['2015-01-01':'2015-01-03']['day_pos'].plot(ax = ax[1], title='day_pos (diurnal cycle)',
                                              xlabel='Дата', ylabel='day_pos (косинус фазы суток)')
ax[0].grid (alpha=0.3)
ax[1].grid (alpha=0.3)


## Обучение и проверка модели

Для начала посмотрим на оценки, если обучать и проверять модель на одних и тех же данных.

Функция `evaluate_model` считает прогноз и рисует диаграмму «наблюдённое — предсказанное»: плотность точек показана шестиугольными корзинами, цвет — число наблюдений, а линия $y = x$ соответствует идеальному прогнозу. В заголовке выводятся RMSE и $R^2$.

Модель собирается в `Pipeline`: сначала `StandardScaler` (стандартизация признаков), затем регрессор. Целевая переменная — температура воздуха `ta`, признаки — все остальные столбцы таблицы.

Такая оценка всегда завышена: модель уже «видела» правильные ответы. Поэтому дальше переходим к кросс-валидации.

**Инструменты:** [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [matplotlib.pyplot.hexbin](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hexbin.html), [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [numpy.corrcoef](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html).


In [ ]:
def evaluate_model (model, X, y):
    y_pred = model.predict(X)

    plt.figure()

    plt.hexbin (y, y_pred, mincnt=1, gridsize=50)

    plt.axis('equal')
    ax = plt.gca()
    lims = [
           np.min([ax.get_xlim(), ax.get_ylim()]),  # min of both axes
           np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
           ]
    plt.plot(lims, lims, 'k-', alpha=0.75, zorder=0)  # линия y = x — идеальный прогноз
    plt.xlim(lims)
    plt.ylim(lims)
    plt.grid()
    plt.xlabel('Наблюдённая температура воздуха, °C')
    plt.ylabel('Предсказанная температура воздуха, °C')
    plt.colorbar(label = 'Число наблюдений')

    r = np.corrcoef(y_pred, y)[0,1]
    r2 = r2_score(y, y_pred)
    mse = mean_squared_error(y, y_pred)

    plt.title ('RMSE = %.2f, R2 = %.2f'%(np.sqrt (mse), r2))

target_var = 'ta'

y = df[target_var]
X = df.drop(target_var, axis=1)

model = Pipeline([
                ('scaler', StandardScaler()),
                ('regressor', LinearRegression())
                ])

model.fit(X, y)
evaluate_model (model, X, y)


## Кросс-валидация штатными средствами sklearn №1

Используем функцию [cross_val_score](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score).

Выбор метрик качества (scoring) для этой функции:
- [по имени](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-string-names)
- создание объекта типа Scorer на основе функции расчета метрики качества с помощью функции [make_scorer](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.make_scorer.html#sklearn.metrics.make_scorer)

Выборка делится на 4 части по сезону (`GroupKFold`): наблюдения одного сезона целиком попадают либо в обучение, либо в проверку, поэтому модель не может «подсмотреть» ответы соседних по времени наблюдений. Параметр `groups` передаёт номер сезона для каждого наблюдения.

**Инструменты:** [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html), [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html), [ShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ShuffleSplit.html), [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html).


### Как выглядит разбиение на train и test

Прежде чем сравнивать схемы кросс-валидации, посмотрим, **что именно** каждая из них делает с выборкой. Функция `plot_cv_splits` строит для любой схемы две картинки:

- **слева** — маска по времени: для каждого фолда синим отмечены наблюдения, попавшие в обучение, белым — в проверку;
- **справа** — доля наблюдений, попавших в проверку, по месяцам года: сразу видно, какие сезоны целиком уходят в тест, а какие присутствуют и в обучении, и в проверке.

Именно эти две картинки объясняют, почему оценки качества у `KFold` и `GroupKFold` по сезонам получаются разными.

**Инструменты:** [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html), [matplotlib.pyplot.pcolormesh](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html), [pandas.DataFrame.groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html), [GroupKFold.split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html#sklearn.model_selection.GroupKFold.split).


In [ ]:
def plot_cv_splits (cv, X, groups = None, title = ''):
    """Показать разбиение выборки на train/test по фолдам.

    Слева — маска по времени: 1 (синий) — наблюдение в обучении, 0 (белый) — в проверке.
    Справа — доля наблюдений, попавших в проверку, по месяцам года.
    """
    is_train = pd.DataFrame(index = X.index)
    for i, (train_index, test_index) in enumerate(cv.split(X, groups = groups)):
        is_train4fold = np.zeros(len(X))
        is_train4fold[train_index] = 1
        is_train[i] = is_train4fold

    fig, ax = plt.subplots (1, 2, figsize=(14, 3.5), gridspec_kw = {'width_ratios': [3, 1]})

    # маска train/test по времени: строка — фолд, синий — обучение, белый — проверка
    ax[0].pcolormesh (X.index, is_train.columns, is_train.T, cmap='Blues', vmin=0, vmax=1)
    ax[0].set_xlabel ('Дата')
    ax[0].set_ylabel ('Номер фолда')
    ax[0].set_yticks (is_train.columns)
    ax[0].set_title (title or 'Разбиение: 1 — обучение (синий), 0 — проверка (белый)')

    # доля проверочных наблюдений по месяцам: 1 — весь месяц в тесте, 0 — весь месяц в обучении
    test_share = pd.DataFrame({i: (1 - is_train[i]).groupby(X.index.month).mean()
                               for i in is_train.columns}).T
    pcm = ax[1].pcolormesh (np.arange(1, 13), test_share.index, test_share.values,
                            cmap='Reds', vmin=0, vmax=1)
    ax[1].set_xlabel ('Месяц')
    ax[1].set_ylabel ('Номер фолда')
    ax[1].set_yticks (test_share.index)
    ax[1].set_xticks (np.arange(1, 13),
                      labels=['янв', 'фев', 'мар', 'апр', 'май', 'июн',
                              'июл', 'авг', 'сен', 'окт', 'ноя', 'дек'],
                      fontsize=7, rotation=90)
    ax[1].set_title ('Доля test по месяцам')
    fig.colorbar (pcm, ax = ax[1], label = 'доля test')


In [ ]:
from sklearn.model_selection import ShuffleSplit, KFold, GroupKFold


cv = ShuffleSplit(n_splits=10, test_size=0.25) #, random_state=0)
#cv = KFold (n_splits=4)

scores = cross_val_score (model, X, y, cv=cv, scoring='neg_root_mean_squared_error') 


print (scores)
print ('score mean: ', -scores.mean()) 
print ('score std: ', scores.std()) 

# KFold без перемешивания делит ряд на 4 непрерывных блока времени
plot_cv_splits (cv, X, title = 'KFold (n_splits=4)')


In [ ]:
from sklearn.metrics import make_scorer, root_mean_squared_error
scorer = make_scorer (root_mean_squared_error)
cv = KFold (n_splits=4)

display(scorer)

scores = cross_val_score (model, X, y, cv=cv, scoring=scorer) 


print (scores)
print ('score mean: ', -scores.mean()) 
print ('score std: ', scores.std()) 

In [ ]:
cv = GroupKFold (n_splits=4)
def month_to_season(month):
    if month in [12, 1, 2]:
        return 1  # Winter
    elif month in [3, 4, 5]:
        return 2  # Spring
    elif month in [6, 7, 8]:
        return 3  # Summer
    elif month in [9, 10, 11]:
        return 4  # Autumn
    else:
        raise ValueError("Month must be between 1 and 12")

seasons = X.index.month.map(month_to_season)
scores = cross_val_score (model, X, y, cv=cv, groups = seasons, scoring='neg_root_mean_squared_error') 

print (scores)
print ('score mean: ', -scores.mean()) 
print ('score std: ', scores.std()) 

# при четырёх группах-сезонах и n_splits=4 каждый фолд целиком оставляет один сезон вне обучения
plot_cv_splits (cv, X, groups = seasons, title = 'GroupKFold (n_splits=4), groups = seasons')


## Кросс-валидация штатными средствами sklearn №2

Теперь используем более продвинутую функцию [cross_validate](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.cross_validate.html)

Функция `cross_validate` умеет больше, чем `cross_val_score`: она считает сразу несколько метрик, возвращает оценки и на тренировочной части (`return_train_score=True`), и сами обученные модели (`return_estimator=True`).

**Инструменты:** [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html).


In [ ]:
cv = KFold ()
scores = cross_validate (model, X, y, cv=cv, scoring=['neg_root_mean_squared_error', 'r2'], return_train_score = True, return_estimator=True)

### Что вернула функция cross_validate

`cross_validate` возвращает **словарь** (`dict`), а не таблицу. Ключи — это метрики и служебные величины, а значения — массивы `numpy.ndarray` длины `n_splits` (по одному элементу на фолд):

- `test_<метрика>` и `train_<метрика>` — значения метрик на проверочной и тренировочной частях: у нас это `test_neg_root_mean_squared_error`, `train_neg_root_mean_squared_error`, `test_r2`, `train_r2`;
- `fit_time` и `score_time` — время обучения и расчёта метрик по фолдам;
- `estimator` — обученные модели (список, появляется при `return_estimator=True`).

Метрики ошибки возвращаются со знаком минус (`neg_root_mean_squared_error`), потому что `scikit-learn` всегда максимизирует метрики. Привычную таблицу «фолды × метрики» можно получить, обернув словарь в `DataFrame`: `pd.DataFrame(scores)`.

**Инструменты:** [cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html), [numpy.ndarray.mean](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.mean.html), [numpy.ndarray.std](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.std.html), [pandas.DataFrame](https://pandas.pydata.org/docs/reference/frame.html).


In [ ]:
display(scores)

In [ ]:
print ('score mean (train): ', -scores['train_neg_root_mean_squared_error'].mean()) 
print ('score mean (test): ',  -scores['test_neg_root_mean_squared_error'].mean()) 

print ('score std (train): ', scores['train_neg_root_mean_squared_error'].std()) 
print ('score std (test): ', scores['test_neg_root_mean_squared_error'].std()) 

## Создаем собственный генератор, разделяющий выборку на train/test  

См. подробнее про [генераторы в Python на примере скатерти-самобранки](https://habr.com/ru/articles/560300/)

**Инструменты:** [выражение yield](https://docs.python.org/3/reference/expressions.html#yield-expressions), [PEP 255 — Simple Generators](https://peps.python.org/pep-0255/).


### Пример генератора в Python

Генератор — это функция с оператором `yield`: она не возвращает результат целиком, а «выдаёт» значения по одному при каждом вызове `next()`. Разбиения в `scikit-learn` (`KFold`, `GroupKFold` и другие) устроены так же: их метод `split` — генератор, который по очереди выдаёт пары индексов `(train, test)`.


In [ ]:
def my_range (n):
    for i in range(n):
        yield i

x = my_range (10) #np.arange (10)
display(f'x is {x}')
display(f'type(x) is {type(x)}')

print (next(x))
print (next(x))

# for i in x:
#     print (i)



### Новый класс для кросс-валидации

Класс `BlockedKFold` реализует блочную кросс-валидацию для временных рядов: выборка режется на блоки по `block_size` наблюдений, каждый блок делится на `n_splits` частей, и на фолде с номером `i` тестовой становится `i`-я часть **каждого** блока. Так проверочные наблюдения оказываются распределены по всему временному ряду, но не соседствуют с обучающими внутри блока — это ослабляет «подсматривание» между соседними по времени наблюдениями.

Класс повторяет интерфейс разбиений `scikit-learn`: метод `get_n_splits` возвращает число фолдов, а `split` — генератор пар индексов `(train, test)`.

**Инструменты:** [интерфейс кросс-валидаторов в sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html), [numpy.arange](https://numpy.org/doc/stable/reference/generated/numpy.arange.html).


In [ ]:
class BlockedKFold:
    def __init__(self, n_splits=3, block_size = 500):
        self.n_splits = n_splits
        self.block_size = block_size
        self.splits = None
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits
    def split(self, X, y=None, groups=None):
        n_samples = len(X)

        fold_size = self.block_size // self.n_splits
        block_start_idx = np.arange(0, n_samples, self.block_size)
        
        for i in range(self.n_splits):
            test_idx = []
            for block_start in block_start_idx:
                start = block_start + i * fold_size
                end = np.min([n_samples-1, block_start + (i + 1) * fold_size])
                test_idx.extend(list(range(start, end)))
            train_idx = list(set(range(n_samples)) - set(test_idx))

            train_idx = np.array(train_idx)
            test_idx = np.array(test_idx)
            
            yield train_idx, test_idx

### Пример использования `BlockedKFold`

Запускаем блочную кросс-валидацию на тех же данных и смотрим на получившееся разбиение — рисуем его той же функцией `plot_cv_splits`.

На левой панели видно, что тестовые наблюдения идут «полосами»: внутри каждого блока это один непрерывный отрезок, а не разбросанные по всему ряду точки. На правой панели видно, что при этом все сезоны присутствуют и в обучении, и в проверке, — в отличие от `GroupKFold` по сезонам.

**Инструменты:** [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html), [numpy.arange](https://numpy.org/doc/stable/reference/generated/numpy.arange.html).


In [ ]:
# пример использования: блочная кросс-валидация для временного ряда
cv = BlockedKFold (n_splits=5, block_size = 10000)

scores = cross_val_score (model, X, y, cv=cv, scoring='neg_root_mean_squared_error')

print (scores)
print ('score mean: ', -scores.mean())
print ('score std: ', scores.std())

plot_cv_splits (cv, X, title = 'BlockedKFold (n_splits=5, block_size = 10000)')
